In [ ]:
import os
import numpy as np
import glob
from collections import defaultdict
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import gsw
from pyproj import Geod
import pandas as pd
import warnings
import dask


In [ ]:
import intake
import xarray as xr
import warnings

# Suppress serialization warnings for deep-time calendars
warnings.simplefilter("ignore", category=xr.SerializationWarning)

# Connect to the catalog once globally
url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
col = intake.open_esm_datastore(url)

target_models = ["ACCESS-CM2", "GISS-E2-2-G"]
models_found = [] # We will use this to build your 'models' set

def get_cloud_datasets(experiment, variable_id):
    """
    Fetches data from the cloud and returns a dict keyed by model name.
    Restricts to a single member_id to prevent merge conflicts.
    """
    print(f"Querying cloud catalog for {variable_id} in {experiment}...")
    
    query = col.search(
        source_id=target_models,
        experiment_id=experiment,
        variable_id=variable_id,
        table_id="Omon",
        member_id="r1i1p1f1"  # <--- THIS IS THE FIX
    )
    
    # Load lazy datasets
    dset_dict = query.to_dataset_dict(
        zarr_kwargs={'consolidated': True},
        xarray_combine_by_coords_kwargs={'compat': 'override', 'join': 'override'}
    )
    
    clean_datasets = {}
    for full_key, ds in dset_dict.items():
        # Extract the model name from the key
        model_name = full_key.split('.')[2]
        
        clean_datasets[model_name] = ds
        models_found.append(model_name)
        
    print(f"  -> Successfully loaded {len(clean_datasets)} models for {variable_id}\n")
    return clean_datasets

sst_datasets = get_cloud_datasets("piControl", "tos")
sss_datasets = get_cloud_datasets("piControl", "sos")
hf_datasets  = get_cloud_datasets("piControl", "hfds")
wf_datasets  = get_cloud_datasets("piControl", "wfo")

models = set(models_found)
print("Final loaded models:", models)

Querying cloud catalog for tos in piControl...

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


  -> Successfully loaded 2 models for tos

Querying cloud catalog for sos in piControl...

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


  -> Successfully loaded 2 models for sos

Querying cloud catalog for hfds in piControl...

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


  -> Successfully loaded 1 models for hfds

Querying cloud catalog for wfo in piControl...

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


  -> Successfully loaded 1 models for wfo

Final loaded models: {'ACCESS-CM2', 'MRI-ESM2-0'}
